<a href="https://colab.research.google.com/github/nitindavegit/Deep-Learning/blob/main/pytorch05_Training_Pipeline_using_nn_module.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [61]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

In [62]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


# Data Preprocessing

In [63]:
df.shape

(569, 33)

In [64]:
df.drop(columns=['id','Unnamed: 32'], inplace=True)
df.shape

(569, 31)

In [65]:
X = df.iloc[:,1:]
y = df.iloc[:,0]

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2)

In [66]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [67]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [68]:
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

# Model creation using nn module of pytorch

In [69]:
class MySimpleNN(nn.Module):
  def __init__(self, num_features):
    super().__init__()
    self.linear = nn.Linear(num_features, 1) # num_features as input and 1 neuron
    self.sigmoid = nn.Sigmoid()

  def forward(self, features):
    out = self.linear(features) # calculating W*X + B
    out = self.sigmoid(out)     # applying sigmoid function on out
    return out

In [70]:
learning_rate = 0.1
epochs = 25

### Using nn module's loss function

In [71]:
# define loss function
loss_function = nn.BCELoss()

### A built-in optimizer from pytorch module

In [72]:
model = MySimpleNN(X_train_tensor.shape[1])

# define an optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# Convert tensors to float32 to match the model's default parameter dtype
X_train_tensor = X_train_tensor.float()
y_train_tensor = y_train_tensor.float()

for epoch in range(epochs):
  # forward pass
  y_pred = model(X_train_tensor)

  # calculating loss
  loss = loss_function(y_pred, y_train_tensor.view(-1,1))

  # clearing gradients
  optimizer.zero_grad()

  # backward pass
  loss.backward()

  # weights and bias upgrade
  optimizer.step()


  print(f"Epoch : {epoch + 1}, loss : {loss.item()}")

Epoch : 1, loss : 0.5085476636886597
Epoch : 2, loss : 0.42321866750717163
Epoch : 3, loss : 0.3697739541530609
Epoch : 4, loss : 0.3326256573200226
Epoch : 5, loss : 0.30502089858055115
Epoch : 6, loss : 0.28353798389434814
Epoch : 7, loss : 0.2662425637245178
Epoch : 8, loss : 0.25195208191871643
Epoch : 9, loss : 0.23989978432655334
Epoch : 10, loss : 0.22956499457359314
Epoch : 11, loss : 0.22058075666427612
Epoch : 12, loss : 0.21268026530742645
Epoch : 13, loss : 0.20566466450691223
Epoch : 14, loss : 0.1993822604417801
Epoch : 15, loss : 0.1937151700258255
Epoch : 16, loss : 0.1885702908039093
Epoch : 17, loss : 0.1838729828596115
Epoch : 18, loss : 0.17956264317035675
Epoch : 19, loss : 0.175589457154274
Epoch : 20, loss : 0.17191211879253387
Epoch : 21, loss : 0.1684960573911667
Epoch : 22, loss : 0.16531194746494293
Epoch : 23, loss : 0.16233499348163605
Epoch : 24, loss : 0.15954381227493286
Epoch : 25, loss : 0.15692006051540375


In [73]:
model.linear.weight

Parameter containing:
tensor([[ 0.2392,  0.2922,  0.0937,  0.3424,  0.0557,  0.2083,  0.0018,  0.2792,
          0.0849, -0.1041,  0.2453,  0.0684,  0.2508, -0.0171,  0.0572, -0.0186,
         -0.0911,  0.1938,  0.0948, -0.1376,  0.3315,  0.0972,  0.3629,  0.1527,
          0.1509,  0.0730,  0.2833,  0.3943,  0.2347,  0.0763]],
       requires_grad=True)

In [74]:
model.linear.bias

Parameter containing:
tensor([-0.2871], requires_grad=True)

In [75]:
X_test_tensor = X_test_tensor.float()
y_test_tensor = y_test_tensor.float()

with torch.no_grad():
  y_pred = model(X_test_tensor)
  y_pred = (y_pred > 0.5 ).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f"accuracy : {accuracy}")

accuracy : 0.5517082214355469
